# Create Pile Foundations - Revit 2025

Notebook template for the `automation-api-pile-foundations` APS Design Automation workflow.

Inputs:
- `input.rvt`
- `pile_foundations.json`

Output:
- `result.rvt`


In [8]:
import os
import json
import logging
import uuid
from pathlib import Path
from dotenv import load_dotenv

from aps_automation_sdk.classes import (
    Activity,
    ActivityInputParameter,
    ActivityOutputParameter,
    AppBundle,
    WorkItem,
)

from aps_automation_sdk.utils import (
    delete_activity,
    delete_appbundle,
    get_token,
    set_nickname,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")


In [9]:
load_dotenv()

CLIENT_ID = os.getenv("CLIENT_ID", "")
CLIENT_SECRET = os.getenv("CLIENT_SECRET", "")

token = get_token(client_id=CLIENT_ID, client_secret=CLIENT_SECRET)
nickname = set_nickname(token, "myUniqueNickNameHere")

print(f"Authentication successful. Nickname: {nickname}")


Authentication successful. Nickname: webinar_nickname


In [10]:
YEAR = 2025

app_bundle_name = f"PileFoundationsDA{YEAR}"
activity_name = f"PileFoundationsActivity{YEAR}"
alias = "dev"
bucket_key = uuid.uuid4().hex

zip_path = Path.cwd() / "2025 - Net8" / "files" / "PileFoundationsDA.bundle.zip"
input_rvt_path = Path.cwd() / "2025 - Net8" / "files" / "ViktorBuilding.rvt"
input_json_path = Path.cwd() / "2025 - Net8" / "files" / "pile_foundations.sample.json"
output_rvt_path = Path.cwd() / "2025 - Net8" / "files" / "output" / "result.rvt"

appbundle_full_alias = f"{nickname}.{app_bundle_name}+{alias}"
activity_full_alias = f"{nickname}.{activity_name}+{alias}"

print(f"App Bundle: {appbundle_full_alias}")
print(f"Activity:   {activity_full_alias}")
print(f"Bundle ZIP: {zip_path}")
print(f"Input RVT:  {input_rvt_path}")
print(f"Input JSON: {input_json_path}")

App Bundle: webinar_nickname.PileFoundationsDA2025+dev
Activity:   webinar_nickname.PileFoundationsActivity2025+dev
Bundle ZIP: c:\Users\AlejandroDuarteVendr\Documents\revit-structural-agent\automation-api-pile-foundations\2025 - Net8\files\PileFoundationsDA.bundle.zip
Input RVT:  c:\Users\AlejandroDuarteVendr\Documents\revit-structural-agent\automation-api-pile-foundations\2025 - Net8\files\ViktorBuilding.rvt
Input JSON: c:\Users\AlejandroDuarteVendr\Documents\revit-structural-agent\automation-api-pile-foundations\2025 - Net8\files\pile_foundations.sample.json


In [11]:
bundle = AppBundle(
    appBundleId=app_bundle_name,
    engine=f"Autodesk.Revit+{YEAR}",
    alias=alias,
    zip_path=str(zip_path),
    description=f"Create pile foundations for Revit {YEAR}",
)

bundle.deploy(token)
print("App bundle deployed successfully!")


App bundle deployed successfully!


In [12]:
input_revit = ActivityInputParameter(
    name="inputModel",
    localName="input.rvt",
    verb="get",
    description="Input Revit model",
    required=True,
    is_engine_input=True,
    bucketKey=bucket_key,
    objectKey="input.rvt",
)

input_json = ActivityInputParameter(
    name="pilePayload",
    localName="pile_foundations.json",
    verb="get",
    description="Pile foundations JSON payload",
    required=True,
    bucketKey=bucket_key,
    objectKey="pile_foundations.json",
)

output_file = ActivityOutputParameter(
    name="resultModel",
    localName="result.rvt",
    verb="put",
    description="Output Revit model",
    bucketKey=bucket_key,
    objectKey="result.rvt",
)

print("Activity parameters defined successfully!")


Activity parameters defined successfully!


In [13]:
activity = Activity(
    id=activity_name,
    parameters=[input_revit, input_json, output_file],
    engine=f"Autodesk.Revit+{YEAR}",
    appbundle_full_name=appbundle_full_alias,
    description=f"Create pile foundations in Revit {YEAR} from JSON",
    alias=alias,
)

activity.set_revit_command_line()
activity.deploy(token=token)
print("Activity deployed successfully!")


Activity deployed successfully!


In [14]:
input_revit.upload_file_to_oss(file_path=str(input_rvt_path), token=token)
input_json.upload_file_to_oss(file_path=str(input_json_path), token=token)
print(f"Uploaded input files: {input_rvt_path.name}, {input_json_path.name}")


Uploaded input files: ViktorBuilding.rvt, pile_foundations.sample.json


In [15]:
work_item = WorkItem(
    parameters=[input_revit, input_json, output_file],
    activity_full_alias=activity_full_alias,
)

print("Work item created. Starting execution...")


Work item created. Starting execution...


In [16]:
status_resp = work_item.execute(token=token, max_wait=900, interval=10)
last_status = status_resp.get("status", "")

print(f"Work item completed with status: {last_status}")
print(json.dumps(status_resp, indent=2))


2026-04-05 20:41:13,130 INFO Polling work item status, id=ece99112340146ac8514d9497cdd48d0
2026-04-05 20:41:13,545 INFO [  0s] status=inprogress report_url=None
2026-04-05 20:41:23,978 INFO [ 10s] status=inprogress report_url=None
2026-04-05 20:41:34,436 INFO [ 20s] status=inprogress report_url=None
2026-04-05 20:41:44,867 INFO [ 30s] status=inprogress report_url=None
2026-04-05 20:41:55,281 INFO [ 40s] status=inprogress report_url=None
2026-04-05 20:42:05,796 INFO [ 50s] status=inprogress report_url=None
2026-04-05 20:42:16,333 INFO [ 60s] status=success report_url=https://dasprod-store.s3.us-east-1.amazonaws.com/workItem/webinar_nickname/ece99112340146ac8514d9497cdd48d0/report?X-Amz-Expires=3600&X-Amz-Security-Token=IQoJb3JpZ2luX2VjEPP%2F%2F%2F%2F%2F%2F%2F%2F%2F%2FwEaCXVzLWVhc3QtMSJHMEUCIEtLlITt81Prq55Dzi%2FCVARDWvsqavxiuoiCMabndV7MAiEA0ILVIGGTGXvnOwyIRuX90fSU62yejYrBS0jlms94F5kqhAQIu%2F%2F%2F%2F%2F%2F%2F%2F%2F%2F%2FARAEGgwyMjA0NzMxNTIzMTAiDHzQxP1MelJD86xeIirYA9%2BTx4gsJctW0V3FUbjYyX

Work item completed with status: success
{
  "status": "success",
  "reportUrl": "https://dasprod-store.s3.us-east-1.amazonaws.com/workItem/webinar_nickname/ece99112340146ac8514d9497cdd48d0/report?X-Amz-Expires=3600&X-Amz-Security-Token=IQoJb3JpZ2luX2VjEPP%2F%2F%2F%2F%2F%2F%2F%2F%2F%2FwEaCXVzLWVhc3QtMSJHMEUCIEtLlITt81Prq55Dzi%2FCVARDWvsqavxiuoiCMabndV7MAiEA0ILVIGGTGXvnOwyIRuX90fSU62yejYrBS0jlms94F5kqhAQIu%2F%2F%2F%2F%2F%2F%2F%2F%2F%2F%2FARAEGgwyMjA0NzMxNTIzMTAiDHzQxP1MelJD86xeIirYA9%2BTx4gsJctW0V3FUbjYyXugjSN1pUJ0n0KzwkW7kgU0F%2FNFcgahwXv%2FkQFY5an1Mawc7aB6U97ESFpdBe4fRMKRHpocX%2FxWs3Bsr6F%2Btu7uKAOvwDW6P%2FU1%2BHlP49ZyL3XDDrbv8hGLxDebLrT%2F4%2FFdG78UzTFCG0bTsTcqwq%2FecS2E1VeCzXHjKTAsnVgDfLEBUWUN4gg2LpfWkCn5%2FKys0BnYYV4GhVay9QayUR82WYHwSvMhUXYYDzXNRx6ed%2FN1l4yjcKU2IERj%2FhRsDvjxDOaEFd1Hg6pdei2FVEif7GbX6EMeF%2B%2BoEvhDv2eKlf7E8pPLJdoAHjbcawTYN6yoRjcqWvD%2FxF0%2F2KdmelnORSPcbYNC4hriAS0gVIjh9V03du72zotXmOrMNSFgT%2ByjphdFa%2F1t%2Fh3dYjuSTKWBxUMW8IuX2jv1Js3ITFd9uGjE6zHP1r%2B6u3YM8g3iGGfEi

In [17]:
if last_status == "success":
    output_rvt_path.parent.mkdir(parents=True, exist_ok=True)
    output_file.download_to(output_path=str(output_rvt_path), token=token)
    print(f"Download successful: {output_rvt_path}")
else:
    print(f"Work item failed or did not complete successfully. Status: {last_status}")


Download successful: c:\Users\AlejandroDuarteVendr\Documents\revit-structural-agent\automation-api-pile-foundations\2025 - Net8\files\output\result.rvt


In [18]:
try:
    delete_activity(activityId=activity_name, token=token)
    print(f"Activity deleted: {activity_full_alias}")
except Exception as exc:
    print(f"Failed to delete activity: {exc}")

try:
    delete_appbundle(appbundleId=app_bundle_name, token=token)
    print(f"App bundle deleted: {app_bundle_name}")
except Exception as exc:
    print(f"Failed to delete app bundle: {exc}")


Activity deleted: webinar_nickname.PileFoundationsActivity2025+dev
App bundle deleted: PileFoundationsDA2025
